# SQL 심화 — CTE와 윈도우 함수  (SQLite 노트북)
원본 파일: `w5_SQL_심화_CTE와_윈도우_함수.sql` · SQL 셀 30개

## 사용법
1. **위에서부터 순서대로** 실행하세요. 앞쪽 셀이 테이블(`customers`, `orders` 등)을 만듭니다. 데이터 생성 셀은 `DROP TABLE IF EXISTS`로 시작해서 다시 실행해도 안전합니다.
2. 연습 문제 셀은 주석 아래에 SQL을 쓰고 실행합니다. 원본 파일에 이미 풀이가 들어 있던 셀은 그대로 가져왔습니다.
3. 준비물: Python 3, `pandas` (`pip install pandas`), VS Code의 Python·Jupyter 확장. `sqlite3`는 파이썬에 내장되어 있어서 설치가 필요 없습니다.

## 원본(BigQuery/DuckDB)에서 자동 변환한 내용
- 테이블 경로 `` `aiffel-515805.modumall.x` ``, `project_name.dataset_name.x` → `x`
- `CREATE OR REPLACE TABLE` → `DROP TABLE IF EXISTS` + `CREATE TABLE`, 열 타입 `STRING`/`INT64`/`DECIMAL`/`DATE` → `TEXT`/`INTEGER`/`REAL`/`TEXT`
- 날짜는 SQLite에 별도 타입이 없어서 `'YYYY-MM-DD'` 텍스트로 저장합니다.

### 셀별 추가 변환
- **C16**: DATE_TRUNC → date(열, 'start of month')
- **C17**: DATE_TRUNC → date(열, 'start of month')
- **C18**: DATE_TRUNC → date(열, 'start of month')
- **C19**: DATE_TRUNC → date(열, 'start of month')
- **C20**: DATE_TRUNC → date(열, 'start of month') / FROM 앞의 마지막 쉼표 제거
- **C23**: ROLLUP/GROUPING → UNION ALL로 직접 구현 (SQLite 미지원)
- **C24**: ROLLUP/GROUPING → UNION ALL로 직접 구현 (SQLite 미지원)
- **C27**: DATE_TRUNC → date(열, 'start of month')
- **C28**: DATE_TRUNC → date(열, 'start of month')
- **C31**: DATE_TRUNC → date(열, 'start of month')

> 검증: 원본 SQL을 DuckDB에서, 변환본을 SQLite에서 각각 실행해 셀별 결과를 비교했습니다. 값이 다른 셀은 없었습니다. (정렬 기준이 없는 쿼리는 행 순서가, 열 이름을 직접 안 붙인 `count(*)` 같은 열은 자동 이름이 엔진마다 다를 수 있습니다.)

In [1]:
import re, sqlite3
import pandas as pd
try:
    from IPython.display import display
except ImportError:          # 일반 파이썬에서 실행할 때
    display = print

con = sqlite3.connect(":memory:")   # 메모리 DB: 노트북(커널)을 끄면 사라집니다

def sql(text):
    """SQL을 실행합니다. SELECT류는 표로 보여주고, CREATE/INSERT 등은 실행만 합니다."""
    stmts, buf = [], ""
    for line in text.splitlines(keepends=True):
        buf += line
        if sqlite3.complete_statement(buf):      # ; 로 끝나는 문장 단위로 분리
            stmts.append(buf.strip()); buf = ""
    if buf.strip():
        stmts.append(buf.strip())
    ran = shown = 0
    for st in stmts:
        body = re.sub(r"--[^\n]*|/\*.*?\*/", "", st, flags=re.S).strip().rstrip(";").strip()
        if not body:
            continue                              # 주석만 있는 조각은 건너뜀
        ran += 1
        if body.split(None, 1)[0].upper() in ("SELECT", "WITH", "PRAGMA", "VALUES"):
            display(pd.read_sql_query(st, con)); shown += 1
        else:
            con.execute(st)
    con.commit()
    if ran == 0:
        print("(아직 작성한 SQL이 없습니다. 주석 아래에 쿼리를 작성한 뒤 다시 실행하세요.)")
    elif shown == 0:
        print("실행 완료")

print("SQLite 버전:", sqlite3.sqlite_version)

SQLite 버전: 3.53.4


## 1. 두 환경의 문법 차이 — 한눈에 보기

> **[C1] 건너뜀** — 원본의 DuckDB/BigQuery용 이름 공간 준비 셀입니다. SQLite는 필요 없습니다.

In [2]:
sql(r"""
-- [C2]
DROP TABLE IF EXISTS customers;
CREATE TABLE customers (
  customer_id TEXT,
  name TEXT,
  country TEXT,
  signup_date TEXT,
  grade TEXT
);

INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze');
""")

실행 완료


In [3]:
sql(r"""
-- [C3]
DROP TABLE IF EXISTS products;
CREATE TABLE products (
  product_id TEXT,
  product_name TEXT,
  category TEXT,
  price REAL
);

INSERT INTO products VALUES
  ('P01', '에어러너', 'Running', 89000),
  ('P02', '클래식 스니커즈', 'Sneakers', 65000),
  ('P03', '첼시 부츠', 'Boots', 145000),
  ('P04', '여름 샌들', 'Sandals', 38000),
  ('P05', '트레일 러너', 'Running', 119000),
  ('P06', '캔버스 스니커즈', 'Sneakers', 49000),
  ('P07', '워커 부츠', 'Boots', 175000),
  ('P08', '슬리퍼 샌들', 'Sandals', 25000),
  ('P09', '양말 세트', 'Accessory', 12000),
  ('P10', '운동화 끈', 'Accessory', 5000);
""")

실행 완료


In [4]:
sql(r"""
-- [C4]
DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
  order_id TEXT,
  customer_id TEXT,
  order_date TEXT,
  status TEXT,
  amount REAL
);

INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000);
""")

실행 완료


In [5]:
sql(r"""
-- [C5]
DROP TABLE IF EXISTS order_item;
CREATE TABLE order_item (
  order_id TEXT,
  product_id TEXT,
  quantity INTEGER,
  unit_price REAL
);

INSERT INTO order_item VALUES
  ('O0001', 'P01', 1, 125000),
  ('O0002', 'P02', 2, 44500),
  ('O0003', 'P03', 1, 45000),
  ('O0004', 'P04', 2, 115000),
  ('O0006', 'P05', 1, 67000),
  ('O0007', 'P06', 2, 79000),
  ('O0008', 'P07', 1, 32000),
  ('O0009', 'P08', 2, 205000),
  ('O0010', 'P09', 1, 99000),
  ('O0011', 'P01', 2, 38000),
  ('O0013', 'P02', 1, 142000),
  ('O0014', 'P03', 2, 44000),
  ('O0015', 'P04', 1, 53000),
  ('O0016', 'P05', 2, 87500),
  ('O0017', 'P06', 1, 61000),
  ('O0018', 'P07', 2, 160000),
  ('O0019', 'P08', 1, 47000),
  ('O0020', 'P09', 2, 107500),
  ('O0021', 'P01', 1, 38000),
  ('O0022', 'P02', 2, 67000),
  ('O0023', 'P03', 1, 92000),
  ('O0024', 'P04', 2, 134000),
  ('O0026', 'P05', 1, 119000),
  ('O0027', 'P06', 2, 202500),
  ('O0028', 'P07', 1, 58000),
  ('O0029', 'P08', 2, 36500),
  ('O0030', 'P09', 1, 187000);
""")

실행 완료


### 2. ▶️ **코드 실행하기 · 코드 셀 6 [C6]**

In [6]:
sql(r"""
-- [C6]
WITH customer_total AS (
  SELECT
    customer_id,
    SUM(amount) AS total_amount
  FROM orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    customer_id
)
SELECT
  *
FROM customer_total
WHERE
  total_amount >= 200000
ORDER BY
  total_amount DESC
""")

,customer_id,total_amount
0,C013,725000.0
1,C007,678000.0
2,C001,380000.0
3,C012,362000.0
4,C003,349000.0
5,C008,314000.0
6,C002,308000.0


### 3. ▶️ **코드 실행하기 · 코드 셀 7 [C7]**

In [7]:
sql(r"""
-- [C7]
WITH customer_total AS (
  SELECT
    customer_id,
    SUM(amount) AS total_amount
  FROM orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    customer_id
), named AS (
  SELECT
    c.name,
    c.grade,
    t.total_amount
  FROM customer_total AS t
  JOIN customers AS c
    ON t.customer_id = c.customer_id
)
SELECT
  *
FROM named
ORDER BY
  total_amount DESC
LIMIT 5
""")

,name,grade,total_amount
0,권준서,Gold,725000.0
1,조은우,Silver,678000.0
2,김민준,Gold,380000.0
3,신아린,Silver,362000.0
4,박도윤,Bronze,349000.0


## 4. ⌨️ 백문이 불여일타 (1)

In [8]:
sql(r"""
-- [C8] ⌨️ 백문이 불여일타 (1) — 상태별 매출 CTE + 50만원 이상 필터

-- [문제]
-- 1) CTE를 사용해, 상태별 매출 합을 구하는 단계(status_revenue)를 만듭니다. (금액이 있는 주문만)
-- 2) 그중 매출이 50만원 이상인 상태만 조회합니다.

with status_revenue as (
  select
    status
    ,sum(amount) as total_amount   -- 별칭 추가
  from orders
  where amount is not null
  group by 1
)

select *
from status_revenue s
where s.total_amount >= 500000   -- 별칭으로 참조
order by s.total_amount desc
""")

,status,total_amount
0,Paid,1909000.0
1,Shipped,1595000.0


## 5. ⌨️ 백문이 불여일타 (2)

In [9]:
sql(r"""
-- [C9] ⌨️ 백문이 불여일타 (2) — 2단계 CTE로 평균 이상 카테고리 찾기

-- [문제]
-- 1) order_items와 products를 JOIN해 카테고리별 매출을 구하는 CTE(cat_rev)를 만듭니다.
-- 2) 그 결과에서 매출이 전체 카테고리 평균보다 높은 카테고리만 조회합니다.

with cat_rev as (
  select 
  p.category
  ,sum(oi.quantity * oi.unit_price) as revenue
  from order_item oi
  join products p
    on oi.product_id = p.product_id
  group by
    p.category
)

select *
from cat_rev
where revenue > (
  select
    avg(revenue)
  from cat_rev
)
order by revenue desc
""")

,category,revenue
0,Sandals,1081000.0
1,Sneakers,989000.0


### 6. ▶️ **코드 실행하기 · 코드 셀 10 [C10]**

In [10]:
sql(r"""
-- [C10]
SELECT
  order_id,
  status,
  amount,
  SUM(amount) OVER (PARTITION BY status) AS status_revenue,
  SUM(amount) OVER () AS total_revenue
FROM orders
WHERE
  amount IS NOT NULL
ORDER BY
  status NULLS LAST,
  amount DESC
""")

,order_id,status,amount,status_revenue,total_revenue
0,O0018,Paid,320000.0,1909000.0,3806000.0
1,O0024,Paid,268000.0,1909000.0,3806000.0
2,O0004,Paid,230000.0,1909000.0,3806000.0
3,O0030,Paid,187000.0,1909000.0,3806000.0
4,O0007,Paid,158000.0,1909000.0,3806000.0
5,O0022,Paid,134000.0,1909000.0,3806000.0
6,O0001,Paid,125000.0,1909000.0,3806000.0
7,O0026,Paid,119000.0,1909000.0,3806000.0
8,O0010,Paid,99000.0,1909000.0,3806000.0
9,O0014,Paid,88000.0,1909000.0,3806000.0


### 7. ▶️ **코드 실행하기 · 코드 셀 11 [C11]**

In [11]:
sql(r"""
-- [C11]
SELECT
  order_id,
  amount,
  ROW_NUMBER() OVER (ORDER BY amount DESC, order_id) AS row_number,
  RANK() OVER (ORDER BY amount DESC) AS rank,
  DENSE_RANK() OVER (ORDER BY amount DESC) AS dense_rank
FROM orders
WHERE
  amount IS NOT NULL
ORDER BY
  amount DESC
LIMIT 8
""")

,order_id,amount,row_number,rank,dense_rank
0,O0009,410000.0,1,1,1
1,O0027,405000.0,2,2,2
2,O0018,320000.0,3,3,3
3,O0024,268000.0,4,4,4
4,O0004,230000.0,5,5,5
5,O0020,215000.0,6,6,6
6,O0030,187000.0,7,7,7
7,O0016,175000.0,8,8,8


### 8. ▶️ **코드 실행하기 · 코드 셀 12 [C12]**

In [12]:
sql(r"""
-- [C12]
WITH product_sales AS (
  SELECT
    p.category,
    p.product_name,
    SUM(oi.quantity * oi.unit_price) AS revenue
  FROM order_item AS oi
  JOIN products AS p
    ON oi.product_id = p.product_id
  GROUP BY
    p.category,
    p.product_name
)
SELECT
  category,
  product_name,
  revenue,
  RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS cat_rank
FROM product_sales
ORDER BY
  category NULLS LAST,
  cat_rank NULLS LAST
""")

,category,product_name,revenue,cat_rank
0,Accessory,양말 세트,501000.0,1
1,Boots,워커 부츠,410000.0,1
2,Boots,첼시 부츠,225000.0,2
3,Running,트레일 러너,361000.0,1
4,Running,에어러너,239000.0,2
5,Sandals,여름 샌들,551000.0,1
6,Sandals,슬리퍼 샌들,530000.0,2
7,Sneakers,캔버스 스니커즈,624000.0,1
8,Sneakers,클래식 스니커즈,365000.0,2


## 9. ⌨️ 백문이 불여일타 (3)

In [13]:
sql(r"""
-- [C13] ⌨️ 백문이 불여일타 (3) — 고객별 총구매액 RANK 상위 5명

-- [문제] 고객별 총구매액(금액 있는 것만)을 구해, 
-- RANK()로 구매액 순위를 매기고 구매액이 큰 고객 5행을 보여주세요. (CTE + 윈도우)

with cus_amount as (
  select c.name as name
  ,sum(amount) as total_amount
  from orders o
  join customers c
    on o.customer_id = c.customer_id
  where amount is not null
  group by c.name
)

select name
 ,total_amount
 ,rank() over (order by total_amount desc) as rnk
 from cus_amount
 order by rnk nulls last, name
 limit 5
""")

,name,total_amount,rnk
0,권준서,725000.0,1
1,조은우,678000.0,2
2,김민준,380000.0,3
3,신아린,362000.0,4
4,박도윤,349000.0,5


## 11. ⌨️ 백문이 불여일타 (4)

In [14]:
sql(r"""
-- [C15] ⌨️ 백문이 불여일타 (4) — DENSE_RANK 순위 + 전체 평균 비교

-- [문제]
-- 1) 고객별 총구매액을 구해 DENSE_RANK로 순위를 매깁니다.
-- 2) 각 고객의 총구매액이 전체 평균 구매액보다 높은지 함께 보여주세요. (CTE + 순위 윈도우 + AVG() OVER ())

with cus_rank as (
  select
    c.name as name,
    sum(o.amount) as total_amount
  from customers c
  join orders o
    on c.customer_id = o.customer_id
  where o.amount is not null
  group by c.name
)

select
  name,
  total_amount,
  rank() over (order by total_amount desc) as rnk,
  case
    when total_amount >= avg(total_amount) over () then 'up'
    else 'down'
  end as vs_avg
from cus_rank
order by rnk;
""")

,name,total_amount,rnk,vs_avg
0,권준서,725000.0,1,up
1,조은우,678000.0,2,up
2,김민준,380000.0,3,up
3,신아린,362000.0,4,up
4,박도윤,349000.0,5,up
5,윤지호,314000.0,6,up
6,이서연,308000.0,7,up
7,최지우,161000.0,8,down
8,한예준,142000.0,9,down
9,정하준,114000.0,10,down


### 12. ▶️ **코드 실행하기 · 코드 셀 16 [C16]**

In [15]:
sql(r"""
-- [C16]
WITH monthly AS (
  SELECT
    date(order_date, 'start of month') AS month,
    SUM(amount) AS revenue
  FROM orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    month
)
SELECT
  month,
  revenue,
  SUM(revenue) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_revenue
FROM monthly
ORDER BY
  month;
""")

,month,revenue,cum_revenue
0,2023-09-01,556000.0,556000.0
1,2023-10-01,775000.0,1331000.0
2,2023-11-01,839000.0,2170000.0
3,2023-12-01,526000.0,2696000.0
4,2024-01-01,850000.0,3546000.0
5,2024-02-01,260000.0,3806000.0


## 13. ⌨️ 백문이 불여일타 (5)

In [16]:
sql(r"""
-- [C17] ⌨️ 백문이 불여일타 (5) — 월별 매출에 전체 평균 나란히 붙이기

-- [문제] 월별 매출에 '전체 평균 매출'을 나란히 붙여, 
-- 각 달이 평균보다 높은지 낮은지 비교할 수 있게 합니다. (AVG(revenue) OVER ())

with avg_revenue as (
  select
    date(order_date, 'start of month') as month,
    sum(amount) as revenue
  from orders
  where amount is not null
  group by 1
)

select
 month,
 revenue,
 avg(revenue) over ()
from avg_revenue
order by month
""")

,month,revenue,avg(revenue) over ()
0,2023-09-01,556000.0,634333.333333
1,2023-10-01,775000.0,634333.333333
2,2023-11-01,839000.0,634333.333333
3,2023-12-01,526000.0,634333.333333
4,2024-01-01,850000.0,634333.333333
5,2024-02-01,260000.0,634333.333333


### 14. ▶️ **코드 실행하기 · 코드 셀 18 [C18]**

In [17]:
sql(r"""
-- [C18]
WITH monthly AS (
  SELECT
    date(order_date, 'start of month') AS month,
    SUM(amount) AS revenue
  FROM orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    month
)
SELECT
  month,
  revenue,
  ROUND(AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 0) AS roll_3mth
FROM monthly
ORDER BY
  month;
""")

,month,revenue,roll_3mth
0,2023-09-01,556000.0,556000.0
1,2023-10-01,775000.0,665500.0
2,2023-11-01,839000.0,723333.0
3,2023-12-01,526000.0,713333.0
4,2024-01-01,850000.0,738333.0
5,2024-02-01,260000.0,545333.0


### 15. ▶️ **코드 실행하기 · 코드 셀 19 [C19]**

In [18]:
sql(r"""
-- [C19]
WITH monthly AS (
  SELECT
    date(order_date, 'start of month') AS month,
    SUM(amount) AS revenue
  FROM orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    month
)
SELECT
  month,
  revenue,
  LAG(revenue) OVER (ORDER BY month) AS bef_revenue,
  revenue - LAG(revenue) OVER (ORDER BY month) AS diff_revenue,
  ROUND(
    (revenue - LAG(revenue) OVER (ORDER BY month)) * 100.0
      / NULLIF(LAG(revenue) OVER (ORDER BY month), 0),
    1
  ) AS diff_revenue_pct
FROM monthly
ORDER BY
  month;
""")

,month,revenue,bef_revenue,diff_revenue,diff_revenue_pct
0,2023-09-01,556000.0,NaN,NaN,NaN
1,2023-10-01,775000.0,556000.0,219000.0,39.4
2,2023-11-01,839000.0,775000.0,64000.0,8.3
3,2023-12-01,526000.0,839000.0,-313000.0,-37.3
4,2024-01-01,850000.0,526000.0,324000.0,61.6
5,2024-02-01,260000.0,850000.0,-590000.0,-69.4


## 16. ⌨️ 백문이 불여일타 (6)

In [19]:
sql(r"""
-- [C20] ⌨️ 백문이 불여일타 (6) — LEAD로 다음 달 매출 붙이기
-- [문제] 위 월별 매출에서 LAG 대신 LEAD를 사용해, 
-- 각 달에 '다음 달 매출'을 나란히 붙입니다. (마지막 달은 NULL이 됩니다.)

WITH monthly AS (
  SELECT
    date(order_date, 'start of month') AS month,
    SUM(amount) AS revenue
  FROM orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    month
)
SELECT
  month,
  revenue,
  LEAD(revenue) OVER (ORDER BY month) AS aft_revenue
FROM monthly
ORDER BY
  month;
""")

,month,revenue,aft_revenue
0,2023-09-01,556000.0,775000.0
1,2023-10-01,775000.0,839000.0
2,2023-11-01,839000.0,526000.0
3,2023-12-01,526000.0,850000.0
4,2024-01-01,850000.0,260000.0
5,2024-02-01,260000.0,NaN


### 17. ▶️ **코드 실행하기 · 코드 셀 21 [C21]**

In [20]:
sql(r"""
-- [C21]
WITH cat AS (
  SELECT
    p.category,
    SUM(oi.quantity * oi.unit_price) AS revenue
  FROM order_item AS oi
  JOIN products AS p
    ON oi.product_id = p.product_id
  GROUP BY
    p.category
)
SELECT
  category,
  revenue,
  FIRST_VALUE(revenue) OVER (ORDER BY revenue DESC) AS max_revenue,
  ROUND(
    revenue * 100.0 / FIRST_VALUE(revenue) OVER (ORDER BY revenue DESC),
    1
  ) AS pct_of_peak
FROM cat
ORDER BY
  revenue DESC
""")

,category,revenue,max_revenue,pct_of_peak
0,Sandals,1081000.0,1081000.0,100.0
1,Sneakers,989000.0,1081000.0,91.5
2,Boots,635000.0,1081000.0,58.7
3,Running,600000.0,1081000.0,55.5
4,Accessory,501000.0,1081000.0,46.3


### 18. ▶️ **코드 실행하기 · 코드 셀 22 [C22]**

In [21]:
sql(r"""
-- [C22]
WITH cust AS (
  SELECT
    customer_id,
    SUM(amount) AS total_spent
  FROM orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    customer_id
)
SELECT
  customer_id,
  total_spent,
  NTILE(4) OVER (ORDER BY total_spent DESC) AS pp_quantile,
  RANK() OVER (ORDER BY total_spent DESC) AS rnk
FROM cust
ORDER BY
  total_spent DESC
""")

,customer_id,total_spent,pp_quantile,rnk
0,C013,725000.0,1,1
1,C007,678000.0,1,2
2,C001,380000.0,1,3
3,C012,362000.0,1,4
4,C003,349000.0,2,5
5,C008,314000.0,2,6
6,C002,308000.0,2,7
7,C004,161000.0,2,8
8,C010,142000.0,3,9
9,C005,114000.0,3,10


### 19. ▶️ **코드 실행하기 · 코드 셀 23 [C23]**

In [22]:
sql(r"""
-- [C23]
-- [SQLite 변환] ROLLUP이 없어서 UNION ALL로 소계·총계 행을 직접 만들었습니다.
WITH base AS (
  SELECT p.category, o.status, oi.quantity * oi.unit_price AS amt
  FROM order_item AS oi
  JOIN products AS p ON oi.product_id = p.product_id
  JOIN orders AS o ON oi.order_id = o.order_id
)
SELECT category, status, SUM(amt) AS revenue
FROM base
GROUP BY category, status
UNION ALL
SELECT category, NULL, SUM(amt)          -- category 소계
FROM base
GROUP BY category
UNION ALL
SELECT NULL, NULL, SUM(amt)              -- 전체 총계
FROM base
ORDER BY category NULLS LAST, status NULLS LAST;
""")

,category,status,revenue
0,Accessory,Paid,286000.0
1,Accessory,Shipped,215000.0
2,Accessory,NaN,501000.0
3,Boots,Paid,466000.0
4,Boots,Placed,32000.0
5,Boots,Returned,45000.0
6,Boots,Shipped,92000.0
7,Boots,NaN,635000.0
8,Running,Paid,320000.0
9,Running,Placed,38000.0


### 20. ▶️ **코드 실행하기 · 코드 셀 24 [C24]**

In [23]:
sql(r"""
-- [C24]
-- [SQLite 변환] ROLLUP이 없어서 UNION ALL로 소계·총계 행을 직접 만들었습니다.
WITH base AS (
  SELECT p.category, oi.quantity * oi.unit_price AS amt
  FROM order_item AS oi
  JOIN products AS p ON oi.product_id = p.product_id
), agg AS (
  SELECT category, 0 AS is_total, SUM(amt) AS revenue, COUNT(*) AS cat_number
  FROM base
  GROUP BY category
  UNION ALL
  SELECT NULL, 1, SUM(amt), COUNT(*)      -- ROLLUP의 총계 행 (GROUPING = 1)
  FROM base
)
SELECT
  CASE
    WHEN is_total = 1 THEN '── 전체 합계'
    WHEN category IS NULL THEN '(카테고리 미입력)'
    ELSE category
  END AS distict,
  revenue,
  cat_number
FROM agg
ORDER BY is_total, category NULLS LAST;
""")

,distict,revenue,cat_number
0,Accessory,501000.0,3
1,Boots,635000.0,6
2,Running,600000.0,6
3,Sandals,1081000.0,6
4,Sneakers,989000.0,6
5,── 전체 합계,3806000.0,27


## 21. 📒 연습 문제 — CTE와 윈도우 함수 (5문항)

In [24]:
sql(r"""
-- [C25] 📒 연습 문제 — CTE와 윈도우 함수 (1/5)

-- 문제: CTE로 고객별 총구매액을 구한 뒤, 30만원 이상인 고객만 조회합니다.

WITH customer_total AS (
  SELECT
    customer_id,
    SUM(amount) AS total_amount
  FROM orders
  WHERE amount IS NOT NULL
  GROUP BY customer_id
)
SELECT *
FROM customer_total
WHERE total_amount >= 300000
ORDER BY total_amount DESC;
""")

,customer_id,total_amount
0,C013,725000.0
1,C007,678000.0
2,C001,380000.0
3,C012,362000.0
4,C003,349000.0
5,C008,314000.0
6,C002,308000.0


In [25]:
sql(r"""
-- [C26] 📒 연습 문제 — CTE와 윈도우 함수 (2/5)

-- 문제: 각 카테고리 안에서 매출 1위 상품만 뽑으세요. (PARTITION BY + ROW_NUMBER)

WITH product_sales AS (
  SELECT
    p.category,
    p.product_name,
    SUM(oi.quantity * oi.unit_price) AS revenue
  FROM order_item AS oi
  JOIN products AS p
    ON oi.product_id = p.product_id
  GROUP BY p.category, p.product_name
),
ranked AS (
  SELECT
    category,
    product_name,
    revenue,
    ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue DESC) AS rn
  FROM product_sales
)
SELECT category, product_name, revenue
FROM ranked
WHERE rn = 1
ORDER BY revenue DESC;
""")

,category,product_name,revenue
0,Sneakers,캔버스 스니커즈,624000.0
1,Sandals,여름 샌들,551000.0
2,Accessory,양말 세트,501000.0
3,Boots,워커 부츠,410000.0
4,Running,트레일 러너,361000.0


In [26]:
sql(r"""
-- [C27] 📒 연습 문제 — CTE와 윈도우 함수 (3/5)

-- 문제: 월별 매출과 누적 매출을 함께 조회합니다.

WITH monthly AS (
  SELECT
    date(order_date, 'start of month') AS month,
    SUM(amount) AS revenue
  FROM orders
  WHERE amount IS NOT NULL
  GROUP BY month
)
SELECT
  month,
  revenue,
  SUM(revenue) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_revenue
FROM monthly
ORDER BY month;
""")

,month,revenue,cum_revenue
0,2023-09-01,556000.0,556000.0
1,2023-10-01,775000.0,1331000.0
2,2023-11-01,839000.0,2170000.0
3,2023-12-01,526000.0,2696000.0
4,2024-01-01,850000.0,3546000.0
5,2024-02-01,260000.0,3806000.0


In [27]:
sql(r"""
-- [C28] 📒 연습 문제 — CTE와 윈도우 함수 (4/5)

-- 문제: 월별 매출에 전월 매출과 증감률(%)을 붙이세요. (LAG)

WITH monthly AS (
  SELECT
    date(order_date, 'start of month') AS month,
    SUM(amount) AS revenue
  FROM orders
  WHERE amount IS NOT NULL
  GROUP BY month
)
SELECT
  month,
  revenue,
  LAG(revenue) OVER (ORDER BY month) AS bef_revenue,
  ROUND(
    (revenue - LAG(revenue) OVER (ORDER BY month)) * 100.0
      / NULLIF(LAG(revenue) OVER (ORDER BY month), 0),
    1
  ) AS diff_revenue_pct
FROM monthly
ORDER BY month;
""")

,month,revenue,bef_revenue,diff_revenue_pct
0,2023-09-01,556000.0,NaN,NaN
1,2023-10-01,775000.0,556000.0,39.4
2,2023-11-01,839000.0,775000.0,8.3
3,2023-12-01,526000.0,839000.0,-37.3
4,2024-01-01,850000.0,526000.0,61.6
5,2024-02-01,260000.0,850000.0,-69.4


In [28]:
sql(r"""
-- [C29] 📒 연습 문제 — CTE와 윈도우 함수 (5/5)

-- 문제: 고객별 총구매액을 NTILE(3)으로 상·중·하 등급으로 나누세요.

WITH cust AS (
  SELECT
    customer_id,
    SUM(amount) AS total_spent
  FROM orders
  WHERE amount IS NOT NULL
  GROUP BY customer_id
)
SELECT
  customer_id,
  total_spent,
  NTILE(3) OVER (ORDER BY total_spent DESC) AS tile,
  CASE NTILE(3) OVER (ORDER BY total_spent DESC)
    WHEN 1 THEN '상'
    WHEN 2 THEN '중'
    ELSE '하'
  END AS grade_label
FROM cust
ORDER BY total_spent DESC;
""")

,customer_id,total_spent,tile,grade_label
0,C013,725000.0,1,상
1,C007,678000.0,1,상
2,C001,380000.0,1,상
3,C012,362000.0,1,상
4,C003,349000.0,1,상
5,C008,314000.0,2,중
6,C002,308000.0,2,중
7,C004,161000.0,2,중
8,C010,142000.0,2,중
9,C005,114000.0,2,중


## 26. 🧪 종합 실습 — 월별 누적 매출과 전월 대비 증감률

In [29]:
sql(r"""
-- [C30] 🧪 종합 실습 — 월별 누적 매출과 전월 대비 증감률 (1/2)

WITH cust AS (
  SELECT
    customer_id,
    SUM(amount) AS total_spent
  FROM orders
  WHERE amount IS NOT NULL
  GROUP BY customer_id
)
SELECT
  customer_id,
  total_spent,
  NTILE(3) OVER (ORDER BY total_spent DESC) AS tile,
  CASE NTILE(3) OVER (ORDER BY total_spent DESC)
    WHEN 1 THEN '상'
    WHEN 2 THEN '중'
    ELSE '하'
  END AS grade_label
FROM cust
ORDER BY total_spent DESC;
""")

,customer_id,total_spent,tile,grade_label
0,C013,725000.0,1,상
1,C007,678000.0,1,상
2,C001,380000.0,1,상
3,C012,362000.0,1,상
4,C003,349000.0,1,상
5,C008,314000.0,2,중
6,C002,308000.0,2,중
7,C004,161000.0,2,중
8,C010,142000.0,2,중
9,C005,114000.0,2,중


## 27. **질문 2.** 월별 매출과 누적매출을 한 그래프에 그려, 성장 추이를 시각화합니다.

In [30]:
sql(r"""
-- [C31] 🧪 종합 실습 — 월별 누적 매출과 전월 대비 증감률 (2/2)

WITH monthly AS (
  SELECT
    date(order_date, 'start of month') AS month,
    SUM(amount) AS revenue
  FROM orders
  WHERE amount IS NOT NULL
  GROUP BY month
)
SELECT
  month,
  revenue,
  SUM(revenue) OVER (ORDER BY month) AS cum_revenue
FROM monthly
ORDER BY month;
""")

,month,revenue,cum_revenue
0,2023-09-01,556000.0,556000.0
1,2023-10-01,775000.0,1331000.0
2,2023-11-01,839000.0,2170000.0
3,2023-12-01,526000.0,2696000.0
4,2024-01-01,850000.0,3546000.0
5,2024-02-01,260000.0,3806000.0


## 28. 코드 퀴즈

In [31]:
sql(r"""
-- [C32] 코드 퀴즈 — 카테고리별 매출 순위 매기기

WITH cat AS (
  SELECT
    p.category,
    SUM(oi.quantity * oi.unit_price) AS revenue
  FROM order_item AS oi
  JOIN products AS p
    ON oi.product_id = p.product_id
  GROUP BY p.category
)
SELECT
  category,
  revenue,
  RANK() OVER (ORDER BY revenue DESC) AS rnk
FROM cat
ORDER BY rnk;
""")

,category,revenue,rnk
0,Sandals,1081000.0,1
1,Sneakers,989000.0,2
2,Boots,635000.0,3
3,Running,600000.0,4
4,Accessory,501000.0,5
